In [19]:
import pandas as pd
import matplotlib.pyplot as plt
import itertools
import numpy as np


from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import KFold
from sklearn.metrics import r2_score, root_mean_squared_error

#import rdkit library for generating cheminformatics descriptors

from rdkit import Chem
from rdkit.Chem import Descriptors

In [20]:
#!gdown 1ZYfvPAt19d7oHvhnleFEXq4vpRA2vv6H -O train.csv
#!gdown 1QU-FyffByeU5w2lYzBKjkOYt0wgUcXsx -O test.csv

train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")


In [21]:
train_tg = train[train['target_type'] == 'tg']
train_egc = train[train['target_type'] == 'egc']

test_tg = test[test['target_type'] == 'tg']
test_egc = test[test['target_type'] == 'egc']

In [22]:

def calculate_descriptors(data):
    desc_dict = []
    for smile in data['smiles']:
        mol = Chem.MolFromSmiles(smile)
        desc = Descriptors.CalcMolDescriptors(mol)
        desc_dict.append(desc)
    return pd.DataFrame(desc_dict)
    

In [23]:
tg_feat_data = calculate_descriptors(train_tg)
egc_feat_data = calculate_descriptors(train_egc)

tg_feat_test = calculate_descriptors(test_tg)
egc_feat_test = calculate_descriptors(test_egc)

In [24]:

tg_feat_data.insert(0,'Polymer_SMILES',train_tg['smiles'].values)
tg_feat_data['target'] = train_tg['target'].values


egc_feat_data.insert(0,'Polymer_SMILES',train_egc['smiles'].values)
egc_feat_data['target'] = train_egc['target'].values


In [25]:
X_tg = tg_feat_data.iloc[:,1:-1]
y_tg = tg_feat_data['target']

X_egc = egc_feat_data.iloc[:,1:-1]
y_egc = egc_feat_data['target']

#Data cleaning

X_tg = X_tg.replace([np.inf, -np.inf], np.nan)
X_tg = X_tg.fillna(np.mean(X_tg))

X_egc = X_egc.replace([np.inf, -np.inf], np.nan)
X_egc = X_egc.fillna(np.mean(X_egc))

In [26]:
#Hyper parameter tuning using cross-validadtion
 
model = Ridge()
 
def cv_lr(X,y):   
    
    
    alpha_values = np.logspace(-10,1,20)
    cv_results = {'alpha':[],'cv_rmse':[]}
    for alpha in alpha_values:
       
        cv_error = run_cv( X, y,alpha)
        cv_results['cv_rmse'].append(cv_error)
        cv_results['alpha'].append(alpha)
  

    cv_results = pd.DataFrame(cv_results)
    cv_results = cv_results.sort_values('cv_rmse')
    
    cv_rmse = cv_results.iloc[0]['cv_rmse']
    alpha_opt = cv_results.iloc[0]['alpha']

    
    scalar = StandardScaler()
    X_norm = scalar.fit_transform(X)

    rrg= Ridge(alpha = alpha_opt)
    model = rrg.fit(X_norm,y)
    train_val = model.predict(X_norm)
    train_error = root_mean_squared_error(y,train_val)
    
    feature_importance = pd.DataFrame({
        'Feature': X.columns,
        'Coefficient': model.coef_,
        'Importance': np.abs(model.coef_)
    })

    feature_importance = feature_importance.sort_values(
        'Importance',
        ascending=False
    )

    return model, cv_rmse, train_error, alpha_opt, scalar, feature_importance
    
  


# 5 fold cross-validation
def run_cv( X, y, alpha):
     
    kf = KFold(n_splits = 5,  shuffle= True, random_state=42)
    errors = []
   
    for idx, (train, val) in enumerate(kf.split(X)):
        X_cv_train = X.values[train]
        X_cv_val = X.values[val]
        
        y_cv_train = y.values[train]
        y_cv_val = y.values[val]
        
        scaler = StandardScaler()
        X_tr_norm = scaler.fit_transform(X_cv_train)
        X_cv_norm = scaler.transform(X_cv_val)
    
        ridge_reg = Ridge(alpha)
        model = ridge_reg.fit(X_tr_norm, y_cv_train)
        y_pred_val = model.predict(X_cv_norm)
        rmse_val = root_mean_squared_error(y_cv_val, y_pred_val)
        errors.append(rmse_val)
        
    mean_rmse = np.mean(errors)

    
    return  mean_rmse

In [27]:
# Training models to identify mportant features 

model_tg, cv_rmse_tg,train_error_tg, alpha_opt_tg, scalar_tg, feats_tg = cv_lr(X_tg,y_tg)
model_egc, cv_rmse_egc,train_error_egc, alpha_opt_egc, scalar_egc, feats_egc = cv_lr(X_egc,y_egc)

In [28]:
# Selecting top Features  

top_feats_tg = feats_tg.sort_values(
    'Importance',
    ascending=False
)['Feature'].head(20).tolist()

top_feats_egc = feats_egc.sort_values(
    'Importance',
    ascending=False
)['Feature'].head(20).tolist()


In [29]:
X_tg_train = X_tg[top_feats_tg]
X_egc_train = X_egc[top_feats_egc]

In [30]:
# Retraining the model with top features

X_tg_train_top = X_tg_train[top_feats_tg]
X_egc_train_top = X_egc_train[top_feats_egc]

model_tg, cv_rmse_tg,train_error_tg, alpha_opt_tg, scalar_tg, feats_tg = cv_lr(X_tg_train_top,y_tg)
model_egc, cv_rmse_egc,train_error_egc, alpha_opt_egc, scalar_egc, feats_egc = cv_lr(X_egc_train_top,y_egc)

In [31]:


X_tg_test = tg_feat_test[top_feats_tg]
X_egc_test = egc_feat_test[top_feats_egc]

#Scaling the test data
X_tg_test_scaled = scalar_tg.transform(X_tg_test)
X_egc_test_scaled = scalar_egc.transform(X_egc_test)

In [32]:
# Model testing

tg_pred = model_tg.predict(X_tg_test_scaled)
egc_pred = model_egc.predict(X_egc_test_scaled)

In [33]:
test_tg['target'] = tg_pred
test_egc['target'] = egc_pred

/tmp/ipykernel_99127/3178606632.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  test_tg['target'] = tg_pred
/tmp/ipykernel_99127/3178606632.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  test_egc['target'] = egc_pred


In [34]:
test_pred = pd.concat([test_tg, test_egc], axis=0)

submission = test_pred[['id','target']]

In [35]:
submission.to_csv('submission.csv', index=False)

In [36]:
# The above model is a baseline model using Ridge regression and cheminformatics descriptors. It can be further improved by exploring other regression algorithms, feature selection techniques, and hyperparameter tuning methods.
# Additionally, incorporating domain knowledge and exploring more advanced molecular representations such as graph-based features or transformer embeddings could potentially enhance the model's performance.
